# 09 Diagnostic Decomposition v1

This stage diagnoses why the current portfolio pipeline underperforms broad benchmarks despite the RL overlay slightly improving on the static allocator.

The core questions are:
- Is RL's value genuinely dynamic state-dependent control, or mostly a better fixed parameter pair?
- Where is performance being lost in the stock-level pipeline?
- Is the bottleneck more likely signal quality, optimizer configuration, or transaction-cost drag?

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
PYTHON = sys.executable
TEST_START = "2020-01"
TEST_END = "2024-11"
COST_BPS = 10.0

PRED_FILE = PROJECT_ROOT / "data/prediction/fm_oos_predictions.parquet"
PANEL_FILE = PROJECT_ROOT / "data/panel/monthly_stock_panel_basic_full.parquet"
RISK_DIR = PROJECT_ROOT / "data/risk/risk_cov_npz"
RISK_META_FILE = PROJECT_ROOT / "data/risk/risk_monthly_metadata.csv"
STATIC_BACKTEST = PROJECT_ROOT / "data/allocator/static_allocator_backtest.csv"
STATIC_WEIGHTS = PROJECT_ROOT / "data/allocator/static_allocator_weights.parquet"
RL_BACKTEST = PROJECT_ROOT / "data/rl_overlay_sac/test_backtest.csv"
RL_WEIGHTS = PROJECT_ROOT / "data/rl_overlay_sac/test_weights.parquet"
RL_ACTIONS = PROJECT_ROOT / "data/rl_overlay_sac/test_action_history.csv"
BENCHMARK_SUMMARY = PROJECT_ROOT / "data/benchmark_eval/benchmark_summary.csv"
BENCHMARK_RETURNS = PROJECT_ROOT / "data/benchmark_eval/benchmark_monthly_returns.csv"

STATIC_FIXED_DIR = PROJECT_ROOT / "data/diagnostics/static_fixed_param"
PORTFOLIO_DIAG_DIR = PROJECT_ROOT / "data/diagnostics/portfolio"
SIGNAL_DIAG_DIR = PROJECT_ROOT / "data/diagnostics/signal"

for path in [STATIC_FIXED_DIR, PORTFOLIO_DIAG_DIR, SIGNAL_DIAG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
required_artifacts = {
    "predictions": PRED_FILE,
    "panel": PANEL_FILE,
    "static_backtest": STATIC_BACKTEST,
    "static_weights": STATIC_WEIGHTS,
    "rl_backtest": RL_BACKTEST,
    "rl_weights": RL_WEIGHTS,
    "rl_action_history": RL_ACTIONS,
    "benchmark_summary": BENCHMARK_SUMMARY,
    "benchmark_monthly_returns": BENCHMARK_RETURNS,
}

status = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
        }
        for name, path in required_artifacts.items()
    ]
)
display(status)

missing = status.loc[~status["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required upstream artifacts: {missing}")

## Static Fixed-Parameter Benchmark

This benchmark reruns the same long-only mean-variance allocator, but fixes lambda and tau to the average values chosen by the SAC overlay during the test episode. If this fixed benchmark is close to SAC, the RL gain is probably parameter selection rather than dynamic control.

In [ ]:
subprocess.run(
    [
        PYTHON,
        "scripts/run_static_fixed_parameter_benchmark.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--pred-file",
        str(PRED_FILE),
        "--risk-dir",
        str(RISK_DIR),
        "--risk-meta-file",
        str(RISK_META_FILE),
        "--returns-file",
        str(PANEL_FILE),
        "--action-history-file",
        str(RL_ACTIONS),
        "--outdir",
        str(STATIC_FIXED_DIR),
        "--cost-bps",
        str(COST_BPS),
        "--test-start",
        TEST_START,
        "--test-end",
        TEST_END,
    ],
    check=True,
)

In [ ]:
benchmark_summary = pd.read_csv(BENCHMARK_SUMMARY)
static_fixed_summary = pd.read_csv(STATIC_FIXED_DIR / "static_fixed_summary.csv")

comparison = pd.concat(
    [
        benchmark_summary,
        static_fixed_summary.rename(
            columns={"mean_monthly_turnover": "mean_monthly_turnover"}
        ),
    ],
    ignore_index=True,
    sort=False,
)
strategy_order = [
    "equal_weight",
    "static_allocator",
    "rl_overlay_sac",
    "spy_buy_hold",
    "static_fixed_param",
]
comparison = comparison.set_index("strategy").reindex(strategy_order).reset_index()
display(
    comparison[
        [
            "strategy",
            "annualized_return",
            "annualized_volatility",
            "sharpe_ratio",
            "max_drawdown",
            "cumulative_return",
            "mean_monthly_turnover",
            "n_months",
        ]
    ]
)

## Portfolio Diagnostics

These diagnostics decompose strategy behavior into gross versus net return, cost drag, turnover, holdings count, concentration, and drawdowns. The goal is to see whether underperformance is coming from costs, sparse/concentrated portfolios, or loss episodes.

In [ ]:
portfolio_jobs = [
    ("static_allocator", STATIC_BACKTEST, STATIC_WEIGHTS),
    ("rl_overlay_sac", RL_BACKTEST, RL_WEIGHTS),
    (
        "static_fixed_param",
        STATIC_FIXED_DIR / "static_fixed_backtest.csv",
        STATIC_FIXED_DIR / "static_fixed_weights.parquet",
    ),
]

for strategy_name, backtest_file, weights_file in portfolio_jobs:
    subprocess.run(
        [
            PYTHON,
            "scripts/run_portfolio_diagnostics_v1.py",
            "--strategy-name",
            strategy_name,
            "--backtest-file",
            str(backtest_file),
            "--weights-file",
            str(weights_file),
            "--outdir",
            str(PORTFOLIO_DIAG_DIR),
            "--start-month",
            TEST_START,
            "--end-month",
            TEST_END,
        ],
        check=True,
    )

In [ ]:
portfolio_summaries = []
for strategy_name, _, _ in portfolio_jobs:
    portfolio_summaries.append(
        pd.read_csv(PORTFOLIO_DIAG_DIR / f"{strategy_name}_portfolio_diag_summary.csv")
    )

portfolio_summary = pd.concat(portfolio_summaries, ignore_index=True)
display(
    portfolio_summary[
        [
            "strategy",
            "mean_turnover",
            "mean_cost_drag",
            "mean_n_holdings",
            "mean_effective_n",
            "mean_max_weight",
            "mean_top10_weight_share",
            "max_drawdown",
        ]
    ]
)

## Signal Diagnostics

The prediction layer is evaluated directly using monthly IC, rank IC, top-minus-bottom decile spread, and mu_hat dispersion. This separates whether the optimizer is struggling from whether the expected-return signal is too weak or too compressed.

In [ ]:
subprocess.run(
    [
        PYTHON,
        "scripts/run_signal_diagnostics_v1.py",
        "--pred-file",
        str(PRED_FILE),
        "--outdir",
        str(SIGNAL_DIAG_DIR),
        "--test-start",
        TEST_START,
        "--test-end",
        TEST_END,
    ],
    check=True,
)

In [ ]:
signal_summary = pd.read_csv(SIGNAL_DIAG_DIR / "signal_summary.csv")
display(
    signal_summary[
        [
            "mean_ic",
            "mean_rank_ic",
            "mean_top_bottom_spread",
            "average_mu_std",
            "average_p90_minus_p10",
            "spread_sharpe",
        ]
    ]
)

## Interpretation Notes

- Compare `static_fixed_param` against `rl_overlay_sac`. Similar performance suggests the SAC policy mostly found a better average lambda/tau pair; a material SAC edge suggests dynamic control is adding value.
- If gross returns are acceptable but net returns decay, cost drag and turnover are likely first-order bottlenecks.
- If IC, rank IC, and top-bottom spread are weak or unstable, signal quality is the likely upstream bottleneck.
- If the signal is positive but portfolios are overly concentrated, too sparse, or too conservative, the optimizer configuration is a natural next target.
- The next ablation notebook should test no-cost allocation, lower turnover penalties, max-weight constraints, signal-strength filters, and fixed grids around the SAC-implied lambda/tau pair.

In [ ]:
plots = [
    PROJECT_ROOT / "data/benchmark_eval/cumulative_nav_comparison.png",
    STATIC_FIXED_DIR / "static_fixed_cumret.png",
    PORTFOLIO_DIAG_DIR / "rl_overlay_sac_cost_drag.png",
    PORTFOLIO_DIAG_DIR / "static_allocator_cost_drag.png",
    SIGNAL_DIAG_DIR / "signal_ic_timeseries.png",
    SIGNAL_DIAG_DIR / "signal_top_decile_spread.png",
]

for plot_path in plots:
    if plot_path.exists():
        print(plot_path.relative_to(PROJECT_ROOT))
        display(Image(filename=str(plot_path)))